# Torch NN with Word2Vec

In [6]:
import pandas as pd
import numpy as np
import random

from tqdm import tqdm
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn

import spacy
nlp = spacy.load("en_core_web_sm", enable=["tokenizer"])

In [7]:
df = pd.read_csv('../../../custom_dataset_v2_mini.csv', delimiter=";")
df.head()

,ID,Label,Text,word_count
0,25454,Google,We propose a new global entity disambiguation ...,33
1,5152,Google,Newton's laws and Newtonian mechanics in gener...,43
2,27268,Google,3 April 2017 Last updated at 09:17 BST Demand ...,35
3,29161,Google,Southern California contains a Mediterranean c...,32
4,45531,Google,NASA scientists discovered cyclopropenylidene ...,33


## Tokenizer

In [8]:
def preprocess(text):
  text = text.lower()
  text = text.replace('\n', ' ').replace('.', ' PUNCT ').replace('!', ' PUNCT ').replace('?', ' PUNCT ')
  tokens = [token.text for token in nlp(text)]
  return tokens

In [9]:
df['tokens'] = df.apply(lambda row: preprocess(row['Text']), axis=1)
df.head()

,ID,Label,Text,word_count,tokens
0,25454,Google,We propose a new global entity disambiguation ...,33,"[we, propose, a, new, global, entity, disambig..."
1,5152,Google,Newton's laws and Newtonian mechanics in gener...,43,"[newton, 's, laws, and, newtonian, mechanics, ..."
2,27268,Google,3 April 2017 Last updated at 09:17 BST Demand ...,35,"[3, april, 2017, last, updated, at, 09:17, bst..."
3,29161,Google,Southern California contains a Mediterranean c...,32,"[southern, california, contains, a, mediterran..."
4,45531,Google,NASA scientists discovered cyclopropenylidene ...,33,"[nasa, scientists, discovered, cyclopropenylid..."


In [10]:
corpus = [token for row in df['tokens'] for token in row]
len(corpus)

18528300

## Subsampling

In [11]:
token_freq = Counter(corpus)
total_token_count = sum(token_freq.values())
token_freq_prob = {token: count / total_token_count for token, count in token_freq.items()}

In [12]:
COMMON_TOKENS_THRESHOLD = 1e-5

def subsampling_prob(token):
  return ((token_freq_prob[token] - COMMON_TOKENS_THRESHOLD) / token_freq_prob[token]) - np.sqrt(COMMON_TOKENS_THRESHOLD / token_freq_prob[token])

keep_prob = {token: subsampling_prob(token) for token in token_freq_prob}
keep_prob

{'we': np.float64(0.9367744014108392),
 'propose': np.float64(0.49228516106118436),
 'a': np.float64(0.9762758298256),
 'new': np.float64(0.9152630022357242),
 'global': np.float64(0.7211628643441171),
 'entity': np.float64(-0.6210440486853337),
 'disambiguation': np.float64(-10.324014942044277),
 '(': np.float64(0.9324035909685575),
 'ed': np.float64(-0.6869492499843146),
 ')': np.float64(0.9321744347288932),
 'model': np.float64(0.7679520392579176),
 'based': np.float64(0.8384595502841011),
 'on': np.float64(0.9578897129336854),
 'contextualized': np.float64(-6.415394970020651),
 'embeddings': np.float64(-0.9626825116194394),
 'of': np.float64(0.9767124949960457),
 'words': np.float64(0.6602812025512799),
 'and': np.float64(0.9796603202951865),
 'entities': np.float64(-0.4454308712031771),
 'PUNCT': np.float64(0.9863565747736376),
 ' ': np.float64(0.9844767235918273),
 'our': np.float64(0.9066165006141025),
 'is': np.float64(0.9659993075990735),
 'bidirectional': np.float64(-6.984158

In [13]:
df.head()

,ID,Label,Text,word_count,tokens
0,25454,Google,We propose a new global entity disambiguation ...,33,"[we, propose, a, new, global, entity, disambig..."
1,5152,Google,Newton's laws and Newtonian mechanics in gener...,43,"[newton, 's, laws, and, newtonian, mechanics, ..."
2,27268,Google,3 April 2017 Last updated at 09:17 BST Demand ...,35,"[3, april, 2017, last, updated, at, 09:17, bst..."
3,29161,Google,Southern California contains a Mediterranean c...,32,"[southern, california, contains, a, mediterran..."
4,45531,Google,NASA scientists discovered cyclopropenylidene ...,33,"[nasa, scientists, discovered, cyclopropenylid..."


In [14]:
# Function to subsample a text
def subsample_text(text_tokens, keep_prob):
    return [token for token in text_tokens if random.random() > keep_prob[token]]

# Apply subsampling to the entire corpus
df['tokens_sub'] = df.apply(lambda row: subsample_text(row['tokens'], keep_prob), axis=1)
df.head()

,ID,Label,Text,word_count,tokens,tokens_sub
0,25454,Google,We propose a new global entity disambiguation ...,33,"[we, propose, a, new, global, entity, disambig...","[entity, disambiguation, ed, based, contextual..."
1,5152,Google,Newton's laws and Newtonian mechanics in gener...,43,"[newton, 's, laws, and, newtonian, mechanics, ...","[newton, newtonian, mechanics, developed, desc..."
2,27268,Google,3 April 2017 Last updated at 09:17 BST Demand ...,35,"[3, april, 2017, last, updated, at, 09:17, bst...","[3, april, updated, 09:17, bst, demand, proper..."
3,29161,Google,Southern California contains a Mediterranean c...,32,"[southern, california, contains, a, mediterran...","[southern, california, mediterranean, infreque..."
4,45531,Google,NASA scientists discovered cyclopropenylidene ...,33,"[nasa, scientists, discovered, cyclopropenylid...","[nasa, discovered, cyclopropenylidene, saturn,..."


In [15]:
# Print the original and subsampled corpus for comparison
print("Original corpus:", len([token for row in df['tokens'] for token in row]))
print("Original vocab length:", len(list(set([token for row in df['tokens'] for token in row]))))
print("Subsampled corpus:", len([token for row in df['tokens_sub'] for token in row]))
print("Subsampled vocab length:", len(list(set([token for row in df['tokens_sub'] for token in row]))))

Original corpus: 18528300
Original vocab length: 140934
Subsampled corpus: 5320627
Subsampled vocab length: 140934


## Word2Vec

In [16]:
# Vocabulary
vocab = list(set([token for row in df['tokens_sub'] for token in row]))
vocab[:5]

['blank2', 'beaufont', 'racks', 'sortable', 'fibrinogen']

In [17]:
# Id to token dictionary
id2tok = dict(enumerate(vocab))
print(list(id2tok.items())[:4])

# Token to id dictionary
tok2id = {token: i for (i, token) in id2tok.items()}
print(list(tok2id.items())[:4])

[(0, 'blank2'), (1, 'beaufont'), (2, 'racks'), (3, 'sortable')]
[('blank2', 0), ('beaufont', 1), ('racks', 2), ('sortable', 3)]


In [18]:
# Skip-grams windows
WINDOW_WIDTH = 5

def windowizer(tokens, wsize = WINDOW_WIDTH):
  out = []
  for i, wd in enumerate(tokens):
    target = tok2id[wd]
    window = [
      i + j for j in range(-wsize, wsize + 1, 1) if (i + j >= 0) & (i + j < len(tokens)) & (j != 0)
    ]
    out += [(target, tok2id[tokens[w]]) for w in window]
  return out

In [19]:
df["window"] = df.apply(lambda row: windowizer(row['tokens_sub']), axis=1)
df.head()

,ID,Label,Text,word_count,tokens,tokens_sub,window
0,25454,Google,We propose a new global entity disambiguation ...,33,"[we, propose, a, new, global, entity, disambig...","[entity, disambiguation, ed, based, contextual...","[(32804, 35857), (32804, 8299), (32804, 45213)..."
1,5152,Google,Newton's laws and Newtonian mechanics in gener...,43,"[newton, 's, laws, and, newtonian, mechanics, ...","[newton, newtonian, mechanics, developed, desc...","[(43667, 125870), (43667, 7431), (43667, 10432..."
2,27268,Google,3 April 2017 Last updated at 09:17 BST Demand ...,35,"[3, april, 2017, last, updated, at, 09:17, bst...","[3, april, updated, 09:17, bst, demand, proper...","[(105931, 67236), (105931, 7306), (105931, 688..."
3,29161,Google,Southern California contains a Mediterranean c...,32,"[southern, california, contains, a, mediterran...","[southern, california, mediterranean, infreque...","[(11442, 5420), (11442, 108625), (11442, 84415..."
4,45531,Google,NASA scientists discovered cyclopropenylidene ...,33,"[nasa, scientists, discovered, cyclopropenylid...","[nasa, discovered, cyclopropenylidene, saturn,...","[(74305, 23435), (74305, 119353), (74305, 2954..."


In [20]:
class Word2VecDataset(Dataset):
  def __init__(self, dataset, vocab_size):
    self.vocabsize = vocab_size
    self.dataset = dataset
    self.data = [pair for row in dataset['window'] for pair in row]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    return self.data[idx]

In [21]:
df_final = Word2VecDataset(
    df[['tokens_sub', 'window']],
    vocab_size = len(vocab)
)

df_final

In [22]:
BATCH_SIZE = 100

dataloader = DataLoader(
    df_final,
    batch_size = BATCH_SIZE,
    shuffle = True,
    num_workers = 0,
    pin_memory = True
)

In [23]:
len(dataloader)

480816

In [24]:
EMBED_DIM = 100

class Word2Vec(nn.Module):
  def __init__(self, vocab_size = len(vocab), embedding_size = EMBED_DIM):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_size)
    self.expand = nn.Linear(embedding_size, vocab_size, bias=False)

  def forward(self, input):
    target_embedding = self.embedding(input)
    logits = self.expand(target_embedding)
    return logits

In [25]:
model = Word2Vec()
model

Word2Vec(
  (embedding): Embedding(140934, 100)
  (expand): Linear(in_features=100, out_features=140934, bias=False)
)

In [26]:
if torch.cuda.is_available():
  device = torch.device('cuda:0')
else:
  device = torch.device('cpu')
device

device(type='cpu')

In [27]:
model.to(device)

Word2Vec(
  (embedding): Embedding(140934, 100)
  (expand): Linear(in_features=100, out_features=140934, bias=False)
)

In [28]:
EPOCHS = 5
LEARNING_RATE = 5e-4
PATH = 'C:/Users/cinthia.nagahama.VERT/Documents/UMINHO/2 Semestre/Aprendizagem Profunda/trabalho'
accumulation_steps = 4  # Accumulate gradients over 4 batches

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = LEARNING_RATE)

def train(model, data, running_loss, epochs = 10):
  pbar = tqdm(range(epochs * len(data)))
  
  for epoch in range(epochs):
    # print(f"### {epoch} ###")
    epoch_loss = 0

    for sample_num, (target, context) in enumerate(data):
      # print(sample_num)
      target = target.to(device)
      context = context.to(device)

      optimizer.zero_grad()
      logits = model(input=context)
      loss = loss_fn(logits, target)

      if not sample_num % 10000:
          pbar.set_description(f'loss[{sample_num}] = {loss.item()}')

      epoch_loss += loss.item()
      loss.backward()
      optimizer.step()  # Update weights after accumulation_steps batches
      pbar.update(1)

    epoch_loss /= len(dataloader)
    running_loss.append(epoch_loss)
    torch.save({
      'epoch': epoch,
      'model_state_dict': model.state_dict(),
      'optimizer_state_dict': optimizer.state_dict(),
      'loss': loss
    }, PATH + f"model_{epoch}.pt")

  pbar.close()


In [31]:
running_loss = []

train(model, dataloader, running_loss, EPOCHS)

torch.save(model.state_dict(), f'{PATH}model.pt')

KeyboardInterrupt: 